# Notebook 4 - Layered (Hybrid) Architecture
## Customer Support Agent - Instinct + Habit + Grandmaster

---

### What you will learn

- Why **three-layer architectures** outperform both pure-reactive and pure-LLM agents
- How the **Reactive Layer** handles immediate safety without any reasoning
- How the **Behavioral Layer** runs pre-compiled routines using LangChain tools directly
- How the **Deliberative Layer** engages the full BDI LangChain agent only when needed
- How each layer prints its entry so you can trace exactly which layer handled each message

Use case:
> **E-commerce Customer Support Agent**

**Builds on:** Notebook_3 (BDI + LangChain tools). The same `@tool` functions and `AgentExecutor` from Notebook_3 are reused here as the deliberative layer.

## 1. Why Layered Architecture?

Pure LLM agents are expensive and slow. Pure rule-based systems are fast but limited.

**Layered architectures combine the best of both worlds** by routing each message to the cheapest layer that can handle it.

```
USER MESSAGE
      |
      v
+-----------------------------------------------------+
|  REACTIVE LAYER  -- The Instinct                    |
|  Stimulus -> Response. No reasoning. Microseconds.  |
|  Handles: safety, abuse, static FAQs               |
|  LLM: No   Tools: No                               |
+------------------+----------------------------------+
                   | (no match)
                   v
+-----------------------------------------------------+
|  BEHAVIORAL LAYER  -- The Habit                     |
|  Pre-compiled routines. Fixed tool sequences.       |
|  Handles: known workflows (order lookup, tracking)  |
|  LLM: No   Tools: Yes (called directly, no agent)  |
+------------------+----------------------------------+
                   | (no routine matched)
                   v
+-----------------------------------------------------+
|  DELIBERATIVE LAYER  -- The Grandmaster             |
|  BDI reasoning. LangChain AgentExecutor.            |
|  Handles: complex, ambiguous, emotional messages    |
|  LLM: Yes  Tools: Yes (agent decides sequence)     |
+------------------+----------------------------------+
                   |
                   v
             FINAL RESPONSE
```

## 2. Layer Responsibilities

| Layer | Analogy | Responsibility | LLM | Speed |
|---|---|---|---|---|
| Reactive | Car brakes | Safety, abuse detection, static FAQs | No | Instant |
| Behavioral | Lane change routine | Pre-compiled order workflows | No | Fast |
| Deliberative | Route planner | Complex reasoning, compensation, escalation | Yes | Slow |

In [10]:
import os
import re
import json
import pandas as pd
from dotenv import load_dotenv

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.tools import tool

load_dotenv()

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.0)

# order.csv is in the same directory as this notebook
orders_df = pd.read_csv("order.csv")
print(f"Loaded {len(orders_df)} orders")
print(orders_df.head(3))

Loaded 30 orders
   order_id   user_name                 product_name      status        date
0      5001    John Doe               Wireless Mouse   Delivered  2024-08-15
1      5002  Jane Smith              Gaming Keyboard     Shipped  2024-08-20
2      5003    John Doe  Noise Cancelling Headphones  Processing  2024-08-21


## 3. LangChain Tools

Same `@tool` functions as Notebook_3 -- reused here as the **reactive execution steps** inside the Behavioral and Deliberative layers.

- **Behavioral layer** calls these tools **directly** in a fixed sequence (no LLM)
- **Deliberative layer** lets the LangChain agent **decide** which tools to call and in what order

In [11]:
from typing import Optional

@tool
def fetch_order(order_id: int) -> str:
    """Fetch full order details from the database given an order ID."""
    result = orders_df[orders_df["order_id"] == order_id]
    if result.empty:
        return f"No order found with ID {order_id}."
    return json.dumps(result.iloc[0].to_dict(), default=str)


@tool
def check_shipping_status(order_id: int) -> str:
    """Get a human-readable shipping status message for an order."""
    result = orders_df[orders_df["order_id"] == order_id]
    if result.empty:
        return f"No order found with ID {order_id}."
    status = result.iloc[0]["status"]
    messages = {
        "Delivered":  "Your order has been delivered.",
        "Shipped":    "Your order is currently on the way.",
        "Processing": "Your order is still being prepared and has not shipped yet.",
        "Cancelled":  "Your order has been cancelled.",
    }
    return messages.get(status, "Order status unknown.")


@tool
def offer_compensation(order_id: int) -> str:
    """Apply a 10% discount to the customer's account as compensation for an order issue."""
    result = orders_df[orders_df["order_id"] == order_id]
    name = result.iloc[0]["user_name"] if not result.empty else "the customer"
    return f"10% discount successfully applied to {name}'s account."


@tool
def provide_order_info(order_id: int) -> str:
    """Provide detailed order information: product name, status, and order date."""
    result = orders_df[orders_df["order_id"] == order_id]
    if result.empty:
        return f"No order found with ID {order_id}."
    r = result.iloc[0]
    return f"Order #{r['order_id']} -- {r['product_name']}, Status: {r['status']}, Placed on: {r['date']}."


@tool
def escalate_to_human(order_id: Optional[int] = None) -> str:
    """Escalate a complex or unresolved issue to the human support team. order_id is optional."""
    suffix = f" for order {order_id}" if order_id else ""
    return (
        f"The issue{suffix} has been escalated to the human support team. "
        "A representative will contact the customer within 24 hours."
    )


tools = [fetch_order, check_shipping_status, offer_compensation, provide_order_info, escalate_to_human]

## 4. Reactive Layer -- The Instinct

This is the **lowest and fastest layer**. It operates on a pure stimulus-response model.

- No LLM, no tools, no reasoning
- Returns a response instantly if the message matches a known pattern
- Returns `None` if the message is not its concern -- passing control down to the next layer

Like a car's ABS brakes: it does not reason about *why* the wheel is locking -- it just acts.

In [12]:
ABUSIVE_WORDS = ["idiot", "stupid", "useless", "moron", "awful"]

STATIC_RESPONSES = {
    "refund policy": "Our refund policy: items can be returned within 30 days for a full refund.",
    "working hours": "Our support team is available Monday-Friday, 9 AM-6 PM.",
    "contact": "You can reach us at support@shop.com or call 1-800-SHOP.",
}


def reactive_layer(message: str) -> str | None:
    """Instinct layer: immediate stimulus-response with zero reasoning."""
    print("  Checking: empty message, abusive language, static FAQs")
    msg = message.lower().strip()

    if not msg:
        return "Please type a message so I can help you."

    if any(word in msg for word in ABUSIVE_WORDS):
        return "I understand you're frustrated. I'm connecting you to a human agent right away."

    for keyword, response in STATIC_RESPONSES.items():
        if keyword in msg:
            return response

    return None  # not handled -- pass to behavioral layer

## 5. Behavioral Layer -- The Habit

This **middle layer** handles routine, well-understood workflows.

- Detects a known intent pattern (order ID present + simple informational request)
- Executes a **pre-compiled fixed sequence** of tool calls -- no LLM decides the order
- Fast and deterministic, like a lane-change routine: check mirrors -> signal -> steer

Returns `None` if the message does not match a known routine -- escalating to the deliberative layer.

In [13]:
SIMPLE_INTENT_KEYWORDS = ["where", "track", "status", "update", "when", "arrive", "check", "details"]


def behavioral_layer(message: str) -> str | None:
    """Habit layer: pre-compiled order-lookup routine. No LLM -- tools called directly."""
    print("  Checking: order ID present + simple informational intent")
    msg = message.lower()

    # Needs an order ID to proceed
    match = re.search(r'\b(50\d{2})\b', msg)
    if not match:
        return None

    # Only handles simple informational requests -- not complaints or compensation asks
    if not any(word in msg for word in SIMPLE_INTENT_KEYWORDS):
        return None

    order_id = int(match.group(1))
    print(f"  Detected order ID {order_id} -- running pre-compiled routine")

    # Pre-compiled 2-step routine: fetch -> check status
    print("  Step 1/2 -- fetch_order")
    order_raw = fetch_order.invoke({"order_id": order_id})

    print("  Step 2/2 -- check_shipping_status")
    status_msg = check_shipping_status.invoke({"order_id": order_id})

    try:
        record = json.loads(order_raw)
        name = record.get("user_name", "there")
        product = record.get("product_name", "your item")
    except Exception:
        name, product = "there", "your item"

    return f"Hi {name}! Regarding your {product} (Order #{order_id}): {status_msg}"

## 6. Deliberative Layer -- The Grandmaster

This **highest and slowest layer** handles everything the lower layers could not.

- Full BDI reasoning via the LangChain agent (same as Notebook_3)
- Gemini infers Beliefs, Desires, Intention -- then decides the tool-call plan
- Used for complex, ambiguous, emotional, or multi-step scenarios
- Like a route planner: considers all factors, formulates a strategy, delegates execution to tools

In [14]:
prompt = ChatPromptTemplate.from_messages([
    ("system", """
You are a BDI (Belief-Desire-Intention) customer support agent for an e-commerce platform.

Before calling any tools, reason through three stages:
1. BELIEFS   -- What facts can you infer from the customer's message?
2. DESIRES   -- What does the customer want? List ALL goals, even if there are multiple.
3. INTENTIONS -- Identify EVERY applicable strategy from this list:
   - 'provide_information'  : customer wants order details or status
   - 'offer_compensation'   : customer wants a refund, discount, or fee waived
   - 'escalate'             : issue is too complex or requires human intervention

Then execute ALL identified intentions by calling tools:
1. Always call fetch_order first to get real order data
2. Call check_shipping_status to understand the delivery situation
3. For EACH intention, call the matching tool:
   - provide_information -> call provide_order_info
   - offer_compensation  -> call offer_compensation
   - escalate            -> call escalate_to_human

If the customer asks for both information AND compensation, call BOTH provide_order_info AND offer_compensation.

Compose a single empathetic response that addresses EVERY goal the customer raised.
Never invent order details -- always use what the tools return.
"""),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

bdi_agent = create_tool_calling_agent(llm=llm, tools=tools, prompt=prompt)
agent_executor = AgentExecutor(agent=bdi_agent, tools=tools, verbose=True)

## 7. LayeredAgent -- The Orchestrator

The agent tries each layer in order, cheapest first.
The first layer that can handle the message wins -- lower layers never even run.

In [15]:
class LayeredAgent:

    def handle_message(self, message: str) -> str:

        # Layer 1: Reactive
        print("[REACTIVE LAYER] Entering...")
        result = reactive_layer(message)
        if result:
            print("[REACTIVE LAYER] Handled.")
            return result
        print("[REACTIVE LAYER] No match -- passing to Behavioral Layer.\n")

        # Layer 2: Behavioral
        print("[BEHAVIORAL LAYER] Entering...")
        result = behavioral_layer(message)
        if result:
            print("[BEHAVIORAL LAYER] Handled.")
            return result
        print("[BEHAVIORAL LAYER] No routine matched -- passing to Deliberative Layer.\n")

        # Layer 3: Deliberative
        print("[DELIBERATIVE LAYER] Entering -- engaging BDI reasoning via LangChain agent...")
        result = agent_executor.invoke({"input": message, "chat_history": []})
        print("[DELIBERATIVE LAYER] Handled.")
        return result["output"]

In [16]:
layered_agent = LayeredAgent()

test_messages = [
    # Reactive layer -- static FAQ
    "What is your refund policy?",
    # Reactive layer -- abusive language
    "You idiots lost my package again!",
    # Behavioral layer -- simple order status check
    "Where is my order 5007? Just need a quick update.",
    # Deliberative layer -- no order ID, ambiguous emotion
    "I am extremely unhappy with your service and want this resolved.",
    # Deliberative layer -- order ID present but needs compensation reasoning
    "Order 5005 was cancelled without my consent! I want compensation.",
]

for msg in test_messages:
    print(f"\n{'='*60}")
    print(f"USER: {msg}")
    print("="*60)
    response = layered_agent.handle_message(msg)
    print(f"\nFINAL RESPONSE:\n{response}")


USER: What is your refund policy?
[REACTIVE LAYER] Entering...
  Checking: empty message, abusive language, static FAQs
[REACTIVE LAYER] Handled.

FINAL RESPONSE:
Our refund policy: items can be returned within 30 days for a full refund.

USER: You idiots lost my package again!
[REACTIVE LAYER] Entering...
  Checking: empty message, abusive language, static FAQs
[REACTIVE LAYER] Handled.

FINAL RESPONSE:
I understand you're frustrated. I'm connecting you to a human agent right away.

USER: Where is my order 5007? Just need a quick update.
[REACTIVE LAYER] Entering...
  Checking: empty message, abusive language, static FAQs
[REACTIVE LAYER] No match -- passing to Behavioral Layer.

[BEHAVIORAL LAYER] Entering...
  Checking: order ID present + simple informational intent
  Detected order ID 5007 -- running pre-compiled routine
  Step 1/2 -- fetch_order
  Step 2/2 -- check_shipping_status
[BEHAVIORAL LAYER] Handled.

FINAL RESPONSE:
Hi Chris Johnson! Regarding your Smartwatch (Order #500

## 8. Key Takeaways

- **Most requests never reach Gemini** -- the reactive and behavioral layers handle the majority cheaply
- **Safety always wins** -- reactive layer checks run before any tool or LLM call
- **Behavioral layer reuses the same `@tool` functions** as the deliberative layer, but calls them in a fixed sequence without an LLM -- this is why it is faster and cheaper
- **The deliberative layer is the BDI agent from Notebook_3** -- fully reused here, activated only when needed
- **Print messages trace exactly which layer handled each message** -- making the routing logic visible

### Layer routing summary from test messages

| Message | Layer | Why |
|---|---|---|
| What is your refund policy? | Reactive | Keyword match, static response |
| You idiots lost my package! | Reactive | Abusive language detected |
| Where is my order 5007? | Behavioral | Order ID + simple intent keyword |
| I am extremely unhappy... | Deliberative | No routine matched, needs reasoning |
| Order 5005 cancelled, want compensation | Deliberative | Order ID present but emotional + compensation intent |

## 9. How Layered Architecture Resolves BDI Limitations

The three messages from Notebook_3's limitations section are run here against the Layered Agent.
Watch which layer intercepts each one -- and compare the time taken vs. the BDI agent.

In [17]:
import time

# Same three messages that exposed BDI's limitations in Notebook_3
resolution_messages = [
    # BDI Limitation 1 fix: static FAQ -- should NEVER reach the LLM
    "What is your refund policy?",

    # BDI Limitation 2 fix: abusive language -- should trigger INSTANT escalation
    "You idiots! My order 5003 has still not arrived. This is outrageous!",

    # BDI Limitation 3 fix: blurred intent -- deliberative layer handles the full complexity
    "My order 5007 still has not shipped. Can you tell me when it will arrive AND refund my shipping fee?",
]

resolution_agent = LayeredAgent()

for msg in resolution_messages:
    print(f"\n{'='*60}")
    print(f"USER: {msg}")
    print("="*60)
    start = time.time()
    response = resolution_agent.handle_message(msg)
    elapsed = time.time() - start
    print(f"\nFINAL RESPONSE:\n{response}")
    print(f"\n[TIME TAKEN: {elapsed:.2f}s]")


USER: What is your refund policy?
[REACTIVE LAYER] Entering...
  Checking: empty message, abusive language, static FAQs
[REACTIVE LAYER] Handled.

FINAL RESPONSE:
Our refund policy: items can be returned within 30 days for a full refund.

[TIME TAKEN: 0.00s]

USER: You idiots! My order 5003 has still not arrived. This is outrageous!
[REACTIVE LAYER] Entering...
  Checking: empty message, abusive language, static FAQs
[REACTIVE LAYER] Handled.

FINAL RESPONSE:
I understand you're frustrated. I'm connecting you to a human agent right away.

[TIME TAKEN: 0.00s]

USER: My order 5007 still has not shipped. Can you tell me when it will arrive AND refund my shipping fee?
[REACTIVE LAYER] Entering...
  Checking: empty message, abusive language, static FAQs
[REACTIVE LAYER] No match -- passing to Behavioral Layer.

[BEHAVIORAL LAYER] Entering...
  Checking: order ID present + simple informational intent
  Detected order ID 5007 -- running pre-compiled routine
  Step 1/2 -- fetch_order
  Step 2

### What changed vs. BDI

| Message | BDI (Notebook_3) | Layered (Notebook_4) |
|---|---|---|
| Refund policy | Gemini called, tools invoked, 2-3s | **Reactive layer** -- static text, ~0s, zero LLM cost |
| Abusive + order query | Gemini reasons, offers compensation | **Reactive layer** -- instant escalation, ~0s, zero LLM cost |
| Status + refund fee | Picks ONE intention, ignores the other | **Deliberative layer** -- Gemini handles full complexity with all tools available |

**The key architectural insight:** the Layered Agent does not try to make the LLM smarter.
It simply **stops the LLM from seeing messages it has no business reasoning about**.
The reactive layer acts as a filter -- protecting both latency and cost -- while the deliberative layer retains the full power of BDI for messages that genuinely need it.